In [1]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# System Paths
RESULTS_DIR = Path("../../ofat_results")
OUTPUT_DIR = Path("ofat_analysis")

# Available Metrics:
# pct_satisfied
# voluntary_stays
# city_mean_income
# mean_neighbor_income
# mean_neighbor_income_variance
# mean_rent
# mean_utility
# mean_value
# mean_vision
# rent_income_timescale_ratio
# gini_coefficient
# homeless_fraction
# theil_index
# moran_i
# spatial_entropy
# neighborhood_heterogeneity
# income_mobility_indicator
# segregation_index
# gentrification_indicator

# Array of all target metrics tracked in your Mesa model datacollector CSVs
TARGET_METRICS = [
    "city_mean_income",
    "mean_neighbor_income",
    "mean_neighbor_income_variance",
    "mean_rent",
    "mean_utility",
    "mean_value",
    "rent_income_timescale_ratio",
    "gini_coefficient",
    "theil_index",
    "moran_i",
    "segregation_index",
    "homeless_fraction",
    "spatial_entropy",
    "neighborhood_heterogeneity",
    "income_mobility_indicator",
    "segregation_index",
    "gentrification_indicator"
]

# --- Agent Analysis Dynamics Config ---
TARGET_PARAM_NAME = "neighborhood_radius"
TARGET_PARAM_VALUE = "3"

# Dynamically scan the folders matching this specific parameter scenario
AGENT_FILE_PATTERN = f"../../ofat_results/{TARGET_PARAM_NAME}/{TARGET_PARAM_VALUE}/run_*/agents_run_*.csv"

# Quantile binning settings
NUM_QUANTILES = 5
QUANTILE_LABELS = [f"Q{i+1}" for i in range(NUM_QUANTILES)]

# Visual Styling Configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 300

In [7]:
# Create the visual outputs target directory if it does not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Discover all generated model run CSV files
csv_files = sorted(list(RESULTS_DIR.glob("**/model_run_*.csv")))
total_files = len(csv_files)

print(f"Status: OK")
print(f"Data Directory: {RESULTS_DIR.resolve()}")
print(f"Target Output Folder: {OUTPUT_DIR.resolve()}")
print(f"Total Simulation Files Found: {total_files}")

Status: OK
Data Directory: /var/home/rick/Studies/Masters/ABM/ABM_Gentrification/ofat_results
Target Output Folder: /var/home/rick/Studies/Masters/ABM/ABM_Gentrification/src/experiments/ofat_analysis
Total Simulation Files Found: 650


In [8]:
raw_data = []

for file_path in csv_files:
    parts = file_path.parts
    param_name = parts[-4]
    param_raw_val = parts[-3]

    try:
        param_value = float(param_raw_val)
    except ValueError:
        param_value = param_raw_val

    try:
        df_run = pd.read_csv(file_path)
        if df_run.empty:
            continue

        # Isolate the final row row
        final_row = df_run.iloc[-1]

        # Build base mapping data payload
        row_payload = {
            "parameter": param_name,
            "value": param_value
        }

        # Inject every requested target metric column state
        for metric in TARGET_METRICS:
            if metric in df_run.columns:
                row_payload[metric] = final_row[metric]

        raw_data.append(row_payload)
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")

# Cast to Master DataFrame
df_master = pd.DataFrame(raw_data)
print(f"Master dataframe loaded with shape: {df_master.shape}")

Master dataframe loaded with shape: (650, 18)


In [9]:
print("--- DATAFRAME PROPERTIES ---")
print(f"Matrix Shape (Rows, Columns): {df_master.shape}\n")

print("--- NULL VALUE COUNT PER METRIC ---")
print(df_master[TARGET_METRICS].isnull().sum(), "\n")

print("--- HEAD PREVIEW ---")
df_master.head()

--- DATAFRAME PROPERTIES ---
Matrix Shape (Rows, Columns): (650, 18)

--- NULL VALUE COUNT PER METRIC ---
city_mean_income                 0
mean_neighbor_income             0
mean_neighbor_income_variance    0
mean_rent                        0
mean_utility                     0
mean_value                       0
rent_income_timescale_ratio      0
gini_coefficient                 0
theil_index                      0
moran_i                          0
segregation_index                0
homeless_fraction                0
spatial_entropy                  0
neighborhood_heterogeneity       0
income_mobility_indicator        0
segregation_index                0
gentrification_indicator         0
dtype: int64 

--- HEAD PREVIEW ---


,parameter,value,city_mean_income,mean_neighbor_income,mean_neighbor_income_variance,mean_rent,mean_utility,mean_value,rent_income_timescale_ratio,gini_coefficient,theil_index,moran_i,segregation_index,homeless_fraction,spatial_entropy,neighborhood_heterogeneity,income_mobility_indicator,gentrification_indicator
0,affordability_share,0.1,6.925407,7.242645,27.452568,0.720586,-0.712720,-0.932399,5.0,0.463104,0.408694,0.664355,0.292607,0.0,0.943305,0.392001,0.035398,0.105283
1,affordability_share,0.1,5.270355,5.457307,9.336194,0.544776,-0.496444,-0.492234,5.0,0.374441,0.254412,0.599866,0.372958,0.0,0.851360,0.344723,0.008696,0.079942
2,affordability_share,0.1,8.873775,9.211900,31.181575,0.900140,-0.861751,-0.908405,5.0,0.436537,0.340111,0.691634,0.322142,0.0,0.816816,0.392960,0.017391,0.133745
3,affordability_share,0.1,11.090223,11.780300,87.647051,1.193697,-0.998858,-1.166700,5.0,0.476164,0.434591,0.569468,0.293103,0.0,0.880331,0.448965,0.008621,0.170587
4,affordability_share,0.1,11.896967,12.022067,51.248246,1.200241,-1.165744,-0.987815,5.0,0.416951,0.308347,0.658673,0.423729,0.0,0.829341,0.400119,0.016949,0.178982


In [10]:
# Group by parameters and their assigned value steps
ofat_summary = (
    df_master.groupby(["parameter", "value"])[TARGET_METRICS]
    .agg(["mean", "std"])
)

# Display the summary table
ofat_summary.head(20)

city_mean_income            mean_neighbor_income  \
                                      mean        std                 mean   
parameter           value                                                    
affordability_share 0.1          10.134236   3.004923            10.689469   
                    0.2          13.823077   4.288649            14.320789   
                    0.3          18.754529   5.473793            18.996819   
                    0.4          21.189900   9.404658            21.590596   
                    0.5          19.610738   7.104618            19.642171   
                    0.6          26.840488  10.753796            26.831128   
                    0.7          23.898268   7.237247            23.885354   
                    0.8          33.709269  13.715342            33.398968   
                    0.9          40.235975  12.054348            40.046214   
                    1.0          33.761725   8.345015            33.352683   
discount_factor_max 0.2          27.785236   9.112000            27.619413   
                    0.4          32.141577   5.833782            31.906226   
                    0.6          30.355828  12.593599            30.151444   
                    0.8          25.904841  12.455622            25.702130   
                    1.0          25.543439   9.164637            25.323937   
                    1.2          26.227275   4.324254            25.993636   
                    1.4          24.121331   6.509344            23.988369   
                    1.6          26.472787  10.243005            26.441332   
                    1.8          33.984843  11.013107            33.676686   
                    2.0          36.621321  16.092072            36.462502   

                                     mean_neighbor_income_variance  \
                                 std                          mean   
parameter           value                                            
affordability_share 0.1     3.221933                    161.200448   
                    0.2     4.290831                    172.596811   
                    0.3     5.528706                    376.466356   
                    0.4     9.602673                    475.945869   
                    0.5     7.185795                    229.377554   
                    0.6    10.788386                    578.208146   
                    0.7     7.392913                    362.811474   
                    0.8    13.655468                   1522.612407   
                    0.9    12.079735                   1074.370151   
                    1.0     8.221046                    671.619822   
discount_factor_max 0.2     9.031495                    437.749184   
                    0.4     5.835376                    732.602334   
                    0.6    12.558220                    654.114835   
                    0.8    12.433869                    690.115291   
                    1.0     9.089400                    467.962783   
                    1.2     4.298601                    310.798947   
                    1.4     6.584223                    321.360557   
                    1.6    10.414376                    512.244212   
                    1.8    11.072477                   1062.507862   
                    2.0    16.079205                   3776.220089   

                                        mean_rent            mean_utility  \
                                   std       mean        std         mean   
parameter           value                                                   
affordability_share 0.1     360.969244   1.057515   0.312638    -1.202339   
                    0.2     146.832615   2.840419   0.848780     0.018974   
                    0.3     476.535765   5.577896   1.618119     2.200494   
                    0.4     513.825877   8.477355   3.781315     4.640508   
                    0.5     234.210479   9.634151   3.541819     6.498944   
    

In [11]:
unique_params = df_master["parameter"].unique()
num_plots = len(unique_params)

# Dynamic grid parameters (3 columns wide layout)
cols = 3
rows = (num_plots + cols - 1) // cols

# Loop through every defined metric to create its own isolated multi-panel file
for metric in TARGET_METRICS:
    if metric not in df_master.columns:
        print(f"Skipping visualization for '{metric}': column missing from extracted data context.")
        continue

    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), sharey=False)
    axes = axes.flatten()

    for idx, param in enumerate(unique_params):
        ax = axes[idx]
        param_subset = df_master[df_master["parameter"] == param]

        # Lineplot automatically groups the 10 replicates per step into mean line + 95% CI band
        sns.lineplot(
            data=param_subset,
            x="value",
            y=metric,
            ax=ax,
            marker="o",
            color="#2b5c8f",
            errorbar="ci"
        )

        ax.set_title(f"Sensitivity to:\n{param}", fontsize=11, fontweight='bold')
        ax.set_xlabel("Parameter Step Value", fontsize=9)
        ax.set_ylabel(metric if idx % cols == 0 else "", fontsize=9)

    # Clean up any empty grids
    for empty_idx in range(idx + 1, len(axes)):
        fig.delaxes(axes[empty_idx])

    plt.tight_layout()

    # Construct structured file pathway output descriptor
    metric_image_path = OUTPUT_DIR / f"{metric}.png"
    plt.savefig(metric_image_path, dpi=300)
    plt.close() # Close plot instance to prevent heavy canvas memory leakages in loop execution

    print(f"Generated batch pipeline layout for: {metric_image_path}")

print("\nAll target model metrics processed successfully.")

Generated batch pipeline layout for: ofat_analysis/city_mean_income.png
Generated batch pipeline layout for: ofat_analysis/mean_neighbor_income.png
Generated batch pipeline layout for: ofat_analysis/mean_neighbor_income_variance.png
Generated batch pipeline layout for: ofat_analysis/mean_rent.png
Generated batch pipeline layout for: ofat_analysis/mean_utility.png
Generated batch pipeline layout for: ofat_analysis/mean_value.png
Generated batch pipeline layout for: ofat_analysis/rent_income_timescale_ratio.png
Generated batch pipeline layout for: ofat_analysis/gini_coefficient.png
Generated batch pipeline layout for: ofat_analysis/theil_index.png
Generated batch pipeline layout for: ofat_analysis/moran_i.png
Generated batch pipeline layout for: ofat_analysis/segregation_index.png
Generated batch pipeline layout for: ofat_analysis/homeless_fraction.png
Generated batch pipeline layout for: ofat_analysis/spatial_entropy.png
Generated batch pipeline layout for: ofat_analysis/neighborhood_he

In [ ]:
# Discover and index all available agent file paths across stochastic runs
agent_files = sorted(glob.glob(AGENT_FILE_PATTERN))

print(f"Status: OK")
print(f"Target Pattern: {AGENT_FILE_PATTERN}")
print(f"Found {len(agent_files)} micro-data targets for percentile-group aggregation.")

In [ ]:
all_run_matrices = []

for file_path in agent_files:
    df_agents = pd.read_csv(file_path)

    initial_step = df_agents["Step"].min()
    final_step = df_agents["Step"].max()

    # Isolate boundary states
    df_initial = df_agents[df_agents["Step"] == initial_step][["AgentID", "initial_income"]].copy()
    df_final = df_agents[df_agents["Step"] == final_step][["AgentID", "income"]].copy()

    # Determine initial percentile ranks relative to this specific run
    df_initial["Initial Quantile"] = pd.qcut(
        df_initial["initial_income"], q=NUM_QUANTILES, labels=QUANTILE_LABELS, duplicates="drop"
    )

    # Determine global final quantile thresholds for this run's ending distribution
    df_final["Final Quantile"] = pd.qcut(
        df_final["income"], q=NUM_QUANTILES, labels=QUANTILE_LABELS, duplicates="drop"
    )

    # Track groups across time bounds
    df_mobility = pd.merge(df_initial, df_final, on="AgentID")

    # Construct transition frequency table normalized by row
    run_matrix = pd.crosstab(
        df_mobility["Initial Quantile"], df_mobility["Final Quantile"], normalize="index"
    )

    # Reindex to guarantee full shape dimensions match even if a bin was empty
    run_matrix = run_matrix.reindex(index=QUANTILE_LABELS, columns=QUANTILE_LABELS, fill_value=0.0)
    all_run_matrices.append(run_matrix)

print(f"Processed {len(all_run_matrices)} independent transition configurations.")

In [ ]:
# Stack along a new structural axis and calculate the mean matrix array
stacked_arrays = np.stack([m.values for m in all_run_matrices], axis=0)
mean_array = np.mean(stacked_arrays, axis=0)

df_aggregate_matrix = pd.DataFrame(
    mean_array,
    index=QUANTILE_LABELS,
    columns=QUANTILE_LABELS
)

print("--- AGGREGATED SYSTEM TRANSITION MATRIX ---")
df_aggregate_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

sns.heatmap(
    df_aggregate_matrix,
    annot=True,
    cmap="Blues",
    fmt=".2f",
    cbar_kws={'label': 'Mean Transition Probability'},
    ax=ax
)

ax.invert_yaxis()  # Put Q1 (lowest earners) at the bottom left origin
ax.set_title("Averaged Quantile Transition Matrix (All Seeds Grouped)", fontweight='bold', fontsize=12)
ax.set_xlabel("Final Income Quantile", fontsize=10)
ax.set_ylabel("Initial Income Quantile", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Local Case Study Context ---
# Choose which specific stochastic run seed to isolate for the snapshot comparison
RUN_CHOICE = "6"

# Construct the explicit path descriptor using the parameter context from Cell 2
SINGLE_RUN_FILE = Path(
    f"../../ofat_results/{TARGET_PARAM_NAME}/{TARGET_PARAM_VALUE}/run_{RUN_CHOICE}/agents_run_{RUN_CHOICE}.csv"
)

# Process the targeted case-study dataset
df_single = pd.read_csv(SINGLE_RUN_FILE)
t_init, t_final = df_single["Step"].min(), df_single["Step"].max()

df_case_init = df_single[df_single["Step"] == t_init][["AgentID", "initial_income"]].copy()
df_case_final = df_single[df_single["Step"] == t_final][["AgentID", "income", "satisfied"]].copy()
df_case_mobility = pd.merge(df_case_init, df_case_final, on="AgentID")

# Setup side-by-side single run diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot A: Absolute Mobility Path Tracking
sns.scatterplot(
    data=df_case_mobility, x="initial_income", y="income",
    hue="satisfied", palette="viridis", alpha=0.8, ax=axes[0]
)

# Add y=x identity line
bounds = [min(df_case_mobility["initial_income"]), max(df_case_mobility["initial_income"])]
axes[0].plot(bounds, bounds, "k--", alpha=0.5, label="No Displacement Trajectory")
axes[0].set_title(f"{TARGET_PARAM_NAME}={TARGET_PARAM_VALUE} (Run {RUN_CHOICE}): Micro Income Paths", fontweight="bold")
axes[0].set_xlabel("Initial Income")
axes[0].set_ylabel("Final Income")
axes[0].legend(title="Satisfied at End")

# Plot B: Single-Run Heatmap
df_case_mobility["Init Q"] = pd.qcut(df_case_mobility["initial_income"], q=NUM_QUANTILES, labels=QUANTILE_LABELS)
df_case_mobility["Final Q"] = pd.qcut(df_case_mobility["income"], q=NUM_QUANTILES, labels=QUANTILE_LABELS)
single_matrix = pd.crosstab(
    df_case_mobility["Init Q"], df_case_mobility["Final Q"], normalize="index"
).reindex(index=QUANTILE_LABELS, columns=QUANTILE_LABELS, fill_value=0.0)

sns.heatmap(single_matrix, annot=True, cmap="Purples", fmt=".2f", ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title(f"{TARGET_PARAM_NAME}={TARGET_PARAM_VALUE} (Run {RUN_CHOICE}): Quantile Transitions", fontweight="bold")
axes[1].set_xlabel("Final Quantile")
axes[1].set_ylabel("Initial Quantile")

plt.tight_layout()
plt.show()

In [ ]:
# --- Process Time-Series Trajectories ---
# Use the same SINGLE_RUN_FILE dataset loaded in Cell 12
df_initial_states = df_single[df_single["Step"] == t_init].copy()

# Sort agents to determine thresholds for bottom 10% and top 10%
df_sorted = df_initial_states.sort_values(by="initial_income")
num_agents = len(df_sorted)
k_10_percent = max(1, int(num_agents * 0.10))

# Extract specific AgentID sets
bottom_10_percent_ids = df_sorted.head(k_10_percent)["AgentID"].values
top_10_percent_ids = df_sorted.tail(k_10_percent)["AgentID"].values

# Set up a 3-panel trajectory diagnostic layout
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

# Define global log-safe Y bounds (using positive scale limits to prevent log(0) errors)
y_min_val = max(1e-3, df_single["income"].min())
Y_MIN, Y_MAX = y_min_val * 0.5, df_single["income"].max() * 2.0
COLOR_MAP = "viridis"

# Panel 1: Complete System Population Trajectories
sns.lineplot(
    data=df_single, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.4, ax=axes[0]
)
axes[0].set_title("All Agents Population", fontweight="bold")
axes[0].get_legend().remove()

# Panel 2: Bottom 10% Initial Earner Trajectories
df_bottom_pop = df_single[df_single["AgentID"].isin(bottom_10_percent_ids)]
sns.lineplot(
    data=df_bottom_pop, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.7, ax=axes[1]
)
axes[1].set_title("Bottom 10% Initial Earners", fontweight="bold")
axes[1].get_legend().remove()

# Panel 3: Top 10% Initial Earner Trajectories
df_top_pop = df_single[df_single["AgentID"].isin(top_10_percent_ids)]
sm = plt.cm.ScalarMappable(
    cmap=COLOR_MAP,
    norm=plt.Normalize(df_single["initial_income"].min(), df_single["initial_income"].max())
)

sns.lineplot(
    data=df_top_pop, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.7, ax=axes[2]
)
axes[2].set_title("Top 10% Initial Earners", fontweight="bold")
axes[2].get_legend().remove()

# Format layout, map log transform, and enable minor gridlines for geometric tracking
for ax in axes:
    ax.set_yscale("log")  # Inject logarithmic transformation stream
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_xlabel("Simulation Step", fontsize=10)
    # Major and minor grid lines help track exponential scaling steps
    ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)

axes[0].set_ylabel("Income Dynamics (Log Scale)", fontsize=10)

# Add colorbar matching baseline thresholds
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), pad=0.02, aspect=20)
cbar.set_label("Baseline Initial Income State", fontsize=10)

plt.suptitle(f"Micro Trajectories Log-Scale Diagnostic ({TARGET_PARAM_NAME}={TARGET_PARAM_VALUE} | Run {RUN_CHOICE})", fontsize=14, fontweight="bold", y=1.02)
plt.show()

In [ ]:
# --- Process Time-Series Trajectories ---
# Use the same SINGLE_RUN_FILE dataset loaded in Cell 12
# Sort agents by initial income to isolate the top and bottom brackets
df_initial_states = df_single[df_single["Step"] == t_init].copy()

# Sort agents to determine thresholds for bottom 10% and top 10%
df_sorted = df_initial_states.sort_values(by="initial_income")
num_agents = len(df_sorted)
k_10_percent = max(1, int(num_agents * 0.10))

# Extract specific AgentID sets
bottom_10_percent_ids = df_sorted.head(k_10_percent)["AgentID"].values
top_10_percent_ids = df_sorted.tail(k_10_percent)["AgentID"].values

# Set up a 3-panel trajectory diagnostic layout
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

# Define global Y bounds so you can cleanly compare this figure with other runs later
Y_MIN, Y_MAX = df_single["income"].min() * 0.9, df_single["income"].max() * 1.1
COLOR_MAP = "viridis"

# Panel 1: Complete System Population Trajectories
sns.lineplot(
    data=df_single, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.4, ax=axes[0]
)
axes[0].set_title("All Agents Population", fontweight="bold")
axes[0].get_legend().remove() # Remove heavy colorbar clutter from individual panels

# Panel 2: Bottom 10% Initial Earner Trajectories
df_bottom_pop = df_single[df_single["AgentID"].isin(bottom_10_percent_ids)]
sns.lineplot(
    data=df_bottom_pop, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.7, ax=axes[1]
)
axes[1].set_title("Bottom 10% Initial Earners", fontweight="bold")
axes[1].get_legend().remove()

# Panel 3: Top 10% Initial Earner Trajectories
df_top_pop = df_single[df_single["AgentID"].isin(top_10_percent_ids)]
sm = plt.cm.ScalarMappable(cmap=COLOR_MAP, norm=plt.Normalize(df_single["initial_income"].min(), df_single["initial_income"].max()))

sns.lineplot(
    data=df_top_pop, x="Step", y="income", hue="initial_income",
    units="AgentID", estimator=None, palette=COLOR_MAP, alpha=0.7, ax=axes[2]
)
axes[2].set_title("Top 10% Initial Earners", fontweight="bold")
axes[2].get_legend().remove()

# Format layout, labels, and map a single shared colorbar to the edge of the canvas
for ax in axes:
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_xlabel("Simulation Step", fontsize=10)
axes[0].set_ylabel("Income Dynamics over Time", fontsize=10)

# Add clear colorbar to show baseline tracking context
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), pad=0.02, aspect=20)
cbar.set_label("Baseline Initial Income State", fontsize=10)

plt.suptitle(f"Micro Trajectories Case Study ({TARGET_PARAM_NAME}={TARGET_PARAM_VALUE} | Run {RUN_CHOICE})", fontsize=14, fontweight="bold", y=1.02)
plt.show()

In [ ]:
from matplotlib.ticker import FormatStrFormatter

# 1. Discover all parameter folders (e.g., 'neighborhood_radius', 'vision_radius')
# We assume the directory structure is: ../../ofat_results/{param_name}/{param_value}/run_*/...
all_param_folders = [f for f in Path("../../ofat_results").iterdir() if f.is_dir()]
all_agent_dfs = []

for param_folder in all_param_folders:
    param_name = param_folder.name
    # 2. Get all agent files for this specific parameter
    pattern = str(param_folder / "*/*/agents_run_*.csv")
    agent_files = glob.glob(pattern)

    for file_path in agent_files:
        parts = Path(file_path).parts
        # parts[-3] is the param value, parts[-2] is the run_idx
        param_val = parts[-3]
        run_idx = parts[-2]

        temp_df = pd.read_csv(file_path)
        temp_df["param_name"] = param_name
        temp_df["param_value"] = pd.to_numeric(param_val)
        temp_df["run_idx"] = run_idx

        # Keep only the final step to avoid massive memory bloat
        final_step = temp_df["Step"].max()
        all_agent_dfs.append(temp_df[temp_df["Step"] == final_step])

master_agent_df = pd.concat(all_agent_dfs, ignore_index=True)
print(f"Master Agent DataFrame ready: {master_agent_df.shape[0]} agent samples.")

In [ ]:
# Cell 15: Parameter-to-Parameter Sensitivity Panels
# We create one figure per parameter found in the master_agent_df
parameters = master_agent_df["param_name"].unique()

for param in parameters:
    subset = master_agent_df[master_agent_df["param_name"] == param]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Panel 1: Log-scale Violin
    sns.violinplot(
        data=subset, x="param_value", y="income",
        hue="param_value", palette="viridis", inner="quartile",
        legend=False, ax=axes[0]
    )
    axes[0].set_yscale("log")
    axes[0].set_title(f"{param}: Income Distribution (Log Scale)", fontweight="bold")
    axes[0].xaxis.set_major_formatter(FormatStrFormatter('%.2f')) # Round labels to 2 decimals

    # Panel 2: Density Migration
    sns.kdeplot(
        data=subset, x="income", hue="param_value",
        palette="viridis", fill=True, common_norm=False,
        alpha=0.2, ax=axes[1]
    )
    axes[1].set_xscale("log")
    axes[1].set_title(f"{param}: Income Density Migration", fontweight="bold")

    plt.tight_layout()
    plt.show()